<a href="https://colab.research.google.com/github/HaileyZweedyk/data-mining-assignments/blob/main/CIS335Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Assignment 2: Predictive Modeling for Loan

Understand customer profiles and improve the accuracy of loan approval predictions

Tasks:
- Analyze dataset containing customer demographic and financial information to build a decision tree model that predicts whether a loan application will be approved
- Experiment with different preprocessing methods and model parameters to optimize prediction accuracy

# Imports

In [ ]:
!pip install pandas numpy matplotlib seaborn sklearn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from numpy import ravel
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import Normalizer, StandardScaler, MinMaxScaler, LabelEncoder

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving loan-approval-dataset.csv to loan-approval-dataset.csv


#Initial DataFrame Initialization

takes the Loan Approval Prediction Dataset and converts it to a pandas DataFrame

Attributes:
- Gender
- Married
- Dependents
- Education
- Self_Employed
- ApplicantIncome
- CoapplicantIncome
- LoanAmount
- Loan_Amount_Term
- Credit_History
- Property_Area
- Loan_Status

In [ ]:
df = pd.read_csv('Loan-Approval-Dataset.csv')
df

,Age,Annual_Income,Spending_Score,Credit_Score,Loan_Approval
0,58,95725,6,456,1
1,48,114654,54,314,0
2,34,65773,4,364,1
3,27,97435,54,820,0
4,40,86886,93,643,0
5,58,96803,63,428,0
6,38,61551,18,771,0
7,42,41394,90,362,1
8,30,99092,44,438,0
9,30,33890,34,798,1


#Separate Features From Target

**Target**: Loan_Status


In [ ]:
# Features (Gender, Married, Dependents, Education, Self_Employed, ApplicantIncome, CoapplicantIncome, LoanAmount, Loan_Amount_Term, Credit_History, Property Area) REMOVED: Loan ID, Loan_Status
X = df.iloc[:, 1:-1]

# Target: Loan_Status
y = (df.iloc[:, -1:])

# Encode y to unique integers
encoder = LabelEncoder()
y = encoder.fit_transform(ravel(y))

# Converts all categorical columns to unique integers
X = pd.get_dummies(X, drop_first=False)


#Split Data

In [ ]:
# Split Data: Train 80% Test 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

#Part 1

#Normalization Pipeline

- MinMax
- Z-Score
- None

In [ ]:
pipe_zscore = Pipeline([
    ('scalar', StandardScaler()),
    ('selector', VarianceThreshold()),
    ('classifier', DecisionTreeClassifier())
])

pipe_minmax = Pipeline([
    ('scalar', MinMaxScaler()),
    ('selector', VarianceThreshold()),
    ('classifier', DecisionTreeClassifier())
])

pipe_none = Pipeline([
    ('selector', VarianceThreshold()),
    ('classifier', DecisionTreeClassifier())
])


#Combine into one Pipeline
pipes = {
    "none": pipe_none,
    "zscore": pipe_zscore,
    "minmax": pipe_minmax
}


#Run Decision Tree Classifier Pipeline

In [ ]:
results = []

for name, pipe in pipes.items():
    pipe.fit(X_train, y_train)
    acc = pipe.score(X_test, y_test)
    results.append((name, acc))

df_results = pd.DataFrame(results, columns=["Preprocessing", "Accuracy"])
print(df_results)

  Preprocessing  Accuracy
0          none  0.333333
1        zscore  0.500000
2        minmax  0.333333


#Part 2

#Adjust Depths

Use depths:
- 3
- 5
- 10

To compare the best decision tree

In [ ]:
depths = [3, 5, 10]
splitters = ["best", "random"]

results = []

for pipe in pipes:
  for d in depths:
    for s in splitters:
      model = DecisionTreeClassifier(max_depth=d, splitter=s)
      model.fit(X_train, y_train)
      acc = model.score(X_test, y_test)
      results.append((pipe, d, s, acc))

df_params = pd.DataFrame(results, columns=["Preprocessing","Depth", "Splitter", "Accuracy"])
print(df_params)


   Preprocessing  Depth Splitter  Accuracy
0           none      3     best  0.166667
1           none      3   random  0.166667
2           none      5     best  0.500000
3           none      5   random  0.166667
4           none     10     best  0.333333
5           none     10   random  0.333333
6         zscore      3     best  0.166667
7         zscore      3   random  0.000000
8         zscore      5     best  0.500000
9         zscore      5   random  0.333333
10        zscore     10     best  0.500000
11        zscore     10   random  0.333333
12        minmax      3     best  0.166667
13        minmax      3   random  0.166667
14        minmax      5     best  0.333333
15        minmax      5   random  0.500000
16        minmax     10     best  0.166667
17        minmax     10   random  0.333333


#Averages

Checks the averages of:
- Splitters: Best, Random
- Depth: 3, 5, 10
- Preprocessing: Z-Score, MinMax, None

This data is used to analyze general patterns within the data to better find the best parameters based on

In [ ]:
best_splitter_avg = df_params[df_params["Splitter"] == "best"]["Accuracy"].mean()
rand_splitter_avg = df_params[df_params["Splitter"] == "random"]["Accuracy"].mean()

print(f'\nBest Splitter Average: \n{best_splitter_avg}')
print(f'\nRandom Splittler Average: \n{rand_splitter_avg}')

depth_three_avg = df_params[df_params["Depth"] == 3]["Accuracy"].mean()
depth_five_avg = df_params[df_params["Depth"] == 5]["Accuracy"].mean()
deph_ten_avg = df_params[df_params["Depth"] == 10]["Accuracy"].mean()

print(f'\nDepth 3 Average: \n{depth_three_avg}')
print(f'\nDepth 5 Average: \n{depth_five_avg}')
print(f'\nDepth 10 Average: \n{deph_ten_avg}')

z_score_avg = df_params[df_params["Preprocessing"] == "zscore"]["Accuracy"].mean()
minmax_avg = df_params[df_params["Preprocessing"] == "minmax"]["Accuracy"].mean()
none_avg = df_params[df_params["Preprocessing"] == "none"]["Accuracy"].mean()

print(f'\nZ-Score Average: \n{z_score_avg}')
print(f'\nMinMax Average: \n{minmax_avg}')
print(f'\nNo Preprocessing Average: \n{none_avg}')


Best Splitter Average: 
0.31481481481481477

Random Splittler Average: 
0.2592592592592593

Depth 3 Average: 
0.13888888888888887

Depth 5 Average: 
0.38888888888888884

Depth 10 Average: 
0.3333333333333333

Z-Score Average: 
0.3055555555555555

MinMax Average: 
0.27777777777777773

No Preprocessing Average: 
0.27777777777777773
